In [ ]:
import pandas as pd
import numpy as np

#  1 LOAD 
RAW_FILE = "Faculty_raw.csv"

# Row 0 = short column names
# Row 1 = full question text
# Row 2 = ImportId junk
df_raw = pd.read_csv(RAW_FILE, header=0, skiprows=[1, 2], low_memory=False)

# Optional!?! keep full question text for reference
q_labels = pd.read_csv(RAW_FILE, header=None, nrows=3).iloc[1].to_dict()

# Rename the actual columns from Qualtrics 
position_rename = {
    " ": "consent",
    "Q1": "collection_support_rating",
    " .1": "underrepresented_subjects",
    " .2": "key_resource_for_students",
    "Q2": "course_materials_satisfaction",
    " .3": "first_day_wisdom",
    "Q3": "book_librarian_likelihood",
    " .4": "book_librarian_barrier",
    " .5": "book_librarian_reason",
    "Q4": "research_materials_satisfaction",
    " .6": "unfound_research_resources",
    " .7": "research_materials_explain",
    " .8": "research_access_location",
    "Q5 ": "ill_or_copyright_experience",
    " .9": "ill_or_copyright_experience_detail",
    " .10": "ill_or_copyright_barriers",
    "Q6": "underutilized_resource",
    " .11": "underutilized_resource_impact",
    "Q7_NPS_GROUP": "library_partner_nps_group",
    "Q7": "library_partner_nps_score",
    " .12": "library_value_to_leadership",
    " .13": "name_credentials",
    " .14": "degree_program",
    " .15": "status_or_role",
    "ResponseId": "response_id",
    "RecordedDate": "recorded_date",
    "Duration (in seconds)": "duration_seconds",
    "Progress": "progress_pct",
}

df = df_raw.rename(columns=position_rename).copy()


#  2 REMOVE TEST / INCOMPLETE RESPONSES 
if "Status" in df.columns:
    df = df[df["Status"] != "Survey Preview"].copy()

if "Finished" in df.columns:
    df["Finished"] = df["Finished"].astype(str).str.upper().str.strip()
    df = df[df["Finished"].isin(["TRUE", "1"])].copy()

df = df.reset_index(drop=True)


#  3 DROP USELESS / SENSITIVE COLUMNS 
df = df.dropna(axis=1, how="all")

drop_cols = [
    "IPAddress", "RecipientLastName", "RecipientFirstName",
    "RecipientEmail", "ExternalReference",
    "LocationLatitude", "LocationLongitude",
    "DistributionChannel", "UserLanguage",
    "Q_RecaptchaScore", "Q_DuplicateRespondent"
]

df = df.drop(columns=[c for c in drop_cols if c in df.columns])


#  4. FiX COLUMN TYPES 
for col in ["StartDate", "EndDate", "recorded_date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

if "duration_seconds" in df.columns:
    df["duration_seconds"] = pd.to_numeric(df["duration_seconds"], errors="coerce")

if "progress_pct" in df.columns:
    df["progress_pct"] = pd.to_numeric(df["progress_pct"], errors="coerce")

# fields were exported as labels, map them to numbers
collection_support_map = {
    "1 Poor Alignment": 1,
    "Poor Alignment": 1,
    "2 Fair Alignment": 2,
    "Fair Alignment": 2,
    "3 Good Alignment": 3,
    "Good Alignment": 3,
    "4 Excellent Alignment": 4,
    "Excellent Alignment": 4,
    "2": 2, "3": 3, "4": 4,   # bare numbers just in case
}

course_materials_map = {
    "1 Very Dissatisfied": 1,
    "Very Dissatisfied": 1,
    "2 Dissatisfied": 2,
    "Dissatisfied": 2,
    "3 Neutral": 3,
    "Neutral": 3,
    "4 Satisfied": 4,
    "Satisfied": 4,
    "5 Very Satisfied": 5,
    "Very Satisfied": 5,
    "2": 2, "3": 3, "4": 4, "5": 5,
}

book_librarian_map = {
    "1 Extremely unlikely": 1,
    "Extremely unlikely": 1,
    "2 Unlikely": 2,
    "Unlikely": 2,
    "3 Neutral": 3,
    "Neutral": 3,
    "4 Likely": 4,
    "Likely": 4,
    "5 Extremely likely": 5,
    "Extremely likely": 5,
    "2": 2, "3": 3, "4": 4, "5": 5,
}

research_materials_map = {
    "1 Very dissatisfied": 1,
    "Very dissatisfied": 1,
    "2 Dissatisfied": 2,
    "Dissatisfied": 2,
    "3 Neutral": 3,
    "Neutral": 3,
    "4 Satisfied": 4,
    "Satisfied": 4,
    "5 Very Satisfied": 5,
    "Very Satisfied": 5,
    "2": 2, "3": 3, "4": 4, "5": 5,
}

if "collection_support_rating" in df.columns:
    df["collection_support_rating_num"] = df["collection_support_rating"].map(collection_support_map)

if "course_materials_satisfaction" in df.columns:
    df["course_materials_satisfaction_num"] = df["course_materials_satisfaction"].map(course_materials_map)

if "book_librarian_likelihood" in df.columns:
    df["book_librarian_likelihood_num"] = df["book_librarian_likelihood"].map(book_librarian_map)

if "research_materials_satisfaction" in df.columns:
    df["research_materials_satisfaction_num"] = df["research_materials_satisfaction"].map(research_materials_map)

if "library_partner_nps_score" in df.columns:
    df["library_partner_nps_score"] = pd.to_numeric(df["library_partner_nps_score"], errors="coerce")

# Check for stragglers
for col, num_col in [
    ("collection_support_rating", "collection_support_rating_num"),
    ("course_materials_satisfaction", "course_materials_satisfaction_num"),
    ("book_librarian_likelihood", "book_librarian_likelihood_num"),
    ("research_materials_satisfaction", "research_materials_satisfaction_num"),
]:
    if col in df.columns and num_col in df.columns:
        missed = df[df[col].notna() & df[num_col].isna()][col].unique()
        if len(missed) > 0:
            print(f"Unmapped in {col}: {missed}")
            
#  5 HELPER Cols 
if "library_partner_nps_score" in df.columns:
    def nps_bucket(score):
        if pd.isna(score):
            return np.nan
        elif score >= 9:
            return "Promoter"
        elif score >= 7:
            return "Passive"
        else:
            return "Detractor"

    df["library_partner_nps_group_calc"] = df["library_partner_nps_score"].apply(nps_bucket)

if "duration_seconds" in df.columns:
    df["duration_minutes"] = (df["duration_seconds"] / 60).round(1)

if "recorded_date" in df.columns:
    df["survey_month"] = df["recorded_date"].dt.to_period("M").astype(str)


#  6 split into QUANT + TxT 
quant_cols = [
    "response_id", "recorded_date", "survey_month",
    "duration_minutes", "progress_pct",
    "collection_support_rating_num",
    "course_materials_satisfaction_num",
    "book_librarian_likelihood_num",
    "research_materials_satisfaction_num",
    "library_partner_nps_score",
    "library_partner_nps_group",
    "library_partner_nps_group_calc",
    "degree_program", "status_or_role"
]
quant_cols = [c for c in quant_cols if c in df.columns]
df_quant = df[quant_cols].copy()

text_cols = [
    "response_id",
    "underrepresented_subjects",
    "key_resource_for_students",
    "first_day_wisdom",
    "book_librarian_barrier",
    "book_librarian_reason",
    "unfound_research_resources",
    "research_materials_explain",
    "research_access_location",
    "ill_or_copyright_experience",
    "ill_or_copyright_experience_detail",
    "ill_or_copyright_barriers",
    "underutilized_resource",
    "underutilized_resource_impact",
    "library_value_to_leadership"
]
text_cols = [c for c in text_cols if c in df.columns]
df_text = df[text_cols].copy()


# 7 export
df.to_csv("faculty_clean_full.csv", index=False)
df_quant.to_csv("faculty_quant.csv", index=False)
df_text.to_csv("faculty_text.csv", index=False)

print(f"✓ Cleaned rows: {len(df)}")
print(f"✓ Quant columns: {len(df_quant.columns)}")
print(f"✓ Text columns: {len(df_text.columns)}")

if len(df) == 0:
    print("WARNING: No rows survived filtering. Check Status and Finished.")
else:
    print("\nSample stats for numeric fields:")
    rating_cols = [
        c for c in [
            "collection_support_rating_num",
            "course_materials_satisfaction_num",
            "book_librarian_likelihood_num",
            "research_materials_satisfaction_num",
            "library_partner_nps_score"
        ] if c in df.columns
    ]
    if rating_cols:
        print(df[rating_cols].describe().round(2))

  

✓ Cleaned rows: 8
✓ Quant columns: 14
✓ Text columns: 14

Sample stats for numeric fields:
       collection_support_rating_num  course_materials_satisfaction_num  \
count                           6.00                               6.00   
mean                            2.50                               3.17   
std                             0.55                               0.75   
min                             2.00                               2.00   
25%                             2.00                               3.00   
50%                             2.50                               3.00   
75%                             3.00                               3.75   
max                             3.00                               4.00   

       book_librarian_likelihood_num  research_materials_satisfaction_num  \
count                           6.00                                 6.00   
mean                            3.83                                 3.83   
st